# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [1]:
# your code here
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

# 1. Read the data
df = pd.read_csv("tic-tac-toe.csv")
print(df.shape)
df.head()

(958, 10)


,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
0,x,x,x,x,o,o,x,o,o,True
1,x,x,x,x,o,o,o,x,o,True
2,x,x,x,x,o,o,o,o,x,True
3,x,x,x,x,o,o,o,b,b,True
4,x,x,x,x,o,o,b,o,b,True


In [6]:
# 2. Inspect: is the dataset reliable?
cells = df.columns[:-1]                      # the 9 board positions

print("Missing values:", df.isna().sum().sum())
print("Duplicated rows:", df.duplicated().sum())
print("Unique cell values:", pd.unique(df[cells].values.ravel()))
print(df["class"].value_counts(normalize=True))

# Sanity check based on the rules of the game: X moves first,
# so (#x - #o) must always be 0 or 1 on a legal board
n_x = (df[cells] == "x").sum(axis=1)
n_o = (df[cells] == "o").sum(axis=1)
pd.crosstab(n_x - n_o, df["class"])

Missing values: 0
Duplicated rows: 0
Unique cell values: ['x' 'o' 'b']
class
True     0.653445
False    0.346555
Name: proportion, dtype: float64


class,False,True
row_0,,
0,316,0
1,16,626


In [7]:
# 3. Convert categorical -> numeric
mapping = {"x": 1, "o": -1, "b": 0}
X = df[cells].apply(lambda col: col.map(mapping)).astype("float32")
y = df["class"].astype(int)                  # True -> 1, False -> 0

# 4. Inputs and output are now separated: X (9 features), y (target)

# 5. Normalization: check the scale
X.describe().loc[["mean", "min", "max"]].round(2)

,TL,TM,TR,ML,MM,MR,BL,BM,BR
mean,0.09,0.05,0.09,0.05,0.12,0.05,0.09,0.05,0.09
min,-1.00,-1.00,-1.00,-1.00,-1.00,-1.00,-1.00,-1.00,-1.00
max,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00


## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [8]:
# your code here
tf.keras.utils.set_random_seed(42)           # reproducible weights/shuffling

# 1. Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 2-3. Sequential model: ReLU in hidden layers, softmax output
model = tf.keras.Sequential([
    tf.keras.Input(shape=(9,)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(2, activation="softmax"),   # 2 classes: 0 = not X win, 1 = X win
])

# 4. Compile
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# 5. Fit
history = model.fit(X_train, y_train, epochs=20, batch_size=32,
                    validation_split=0.1, verbose=0)

# 6. Evaluate
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.3f} | Test accuracy: {test_acc:.3f}")

# 7. Save (Keras 3 requires .keras extension)
model.save("tic-tac-toe.keras")

Test loss: 0.209 | Test accuracy: 0.953


## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [9]:
# your code here
loaded_model = tf.keras.models.load_model("tic-tac-toe.keras")

sample = X_test.sample(5, random_state=1)
proba = loaded_model.predict(sample, verbose=0)

pd.DataFrame({
    "actual": y_test.loc[sample.index],
    "predicted": proba.argmax(axis=1),        # class with highest probability
    "P(X wins)": proba[:, 1].round(3),
})

,actual,predicted,P(X wins)
684,0,0,0.367
929,0,0,0.076
46,1,1,0.916
22,1,1,0.758
895,0,0,0.049


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [10]:
# your code here
def build_and_evaluate(hidden_layers, learning_rate, epochs):
    tf.keras.utils.set_random_seed(42)
    model = tf.keras.Sequential([tf.keras.Input(shape=(9,))])
    for n in hidden_layers:
        model.add(tf.keras.layers.Dense(n, activation="relu"))
    model.add(tf.keras.layers.Dense(2, activation="softmax"))

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    model.fit(X_train, y_train, epochs=epochs, batch_size=32,
              validation_split=0.1, verbose=0)
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    return model, round(loss, 3), round(acc, 3)

experiments = [
    ([32, 16],     0.001, 20),    # baseline
    ([32, 16],     0.001, 100),   # more epochs only
    ([32, 16],     0.005, 20),    # higher learning rate only
    ([64, 32, 16], 0.001, 20),    # extra layer only
    ([32, 16],     0.005, 100),   # lr + epochs
    ([64, 32, 16], 0.005, 100),   # everything at once
]

results = []
for layers, lr, ep in experiments:
    _, loss, acc = build_and_evaluate(layers, lr, ep)
    results.append({"layers": layers, "lr": lr, "epochs": ep,
                    "test_loss": loss, "test_acc": acc})
pd.DataFrame(results)

,layers,lr,epochs,test_loss,test_acc
0,"[32, 16]",0.001,20,0.209,0.953
1,"[32, 16]",0.001,100,0.015,0.995
2,"[32, 16]",0.005,20,0.021,0.984
3,"[64, 32, 16]",0.001,20,0.042,0.984
4,"[32, 16]",0.005,100,0.012,0.995
5,"[64, 32, 16]",0.005,100,0.082,0.984


**Which approach(es) did you find helpful to improve your model performance?**

The baseline (2 hidden layers, Adam lr=0.001, 20 epochs) reached 0.953 accuracy but a test loss of 0.209, so it did not meet the target (loss < 0.1, accuracy > 0.95). I changed one hyperparameter at a time to isolate its effect:

- **More epochs (20 → 100)** was the most effective single change (loss 0.015, accuracy 0.995). With only 20 epochs the model was still learning: it was mostly right but not confident.
- **Higher learning rate (0.001 → 0.005)** gave a similar improvement in only 20 epochs (loss 0.021), because each weight update is larger, so the model converges faster.
- **An extra hidden layer** also helped (loss 0.042), but less than the other two.
- **Combining all three** made things *worse* (loss 0.082). The model reached a training loss of 0 and overfit: it became overconfident on the few test boards it got wrong, which cross-entropy penalizes heavily.

Conclusion: the problem was under-training, not lack of model capacity. Training longer or with a larger learning rate was enough; adding layers was unnecessary for a 9-feature problem.